# Readme E-Codices (Handschriften)

Dieses Jupyter Notebook bereitet Datenobjekte und Metadaten aus dem E-Codices-Bestand vor für die digitale Langzeitarchivierung (DLZA) der ZHB Luzern, basierend auf der GOCFL implementierung von Jürgen Enge, https://github.com/je4/gocfl .

Um eine Collection korrekt vorzubereiten, sollten die einzelnen Scripts immer alle in der richtigen Reihenfolge ausgeführt werden.



## 1 - Ordnerstruktur erstellen

Die Scripts in diesem Notebook basieren darauf, dass folgende Ordnerstruktur vorhanden ist, und dass die Ordner jeweils zu Beginn geleert werden.

- Ordner 'files': hier werden diverse Dateien (z.B. eine Textdatei mit allen Signaturen) abgelegt
- Ordner 'info': hier werden die info.json Dateien abgelegt.
- Ordner 'metadata': hier werden die semantischen Metadaten zu den LZA-Objekten abgelegt.
- Ordner 'objects': hier werden die zu archivierenden Datenobjekte (Payload) abgelegt.


In [ ]:
import os
import shutil
from datetime import datetime

# remove existing directories
if os.path.exists('files'):    
    shutil.rmtree('files')

if os.path.exists('info'):
    shutil.rmtree('info')
    
if os.path.exists('metadata'):
    shutil.rmtree('metadata')

# exception: don't remove directory 'objects'
if os.path.exists('objects'):
    pass
else:
    os.mkdir('objects')
 
    
# create necessary directories:
os.mkdir('files')
os.mkdir('info')
os.mkdir('metadata')  

# list all directories in current working directory:
working_directory = './'
print(f"Working directory contains the following directories:")
items = os.listdir(working_directory)
for item in items:
    if os.path.isdir(item):
        print(item)
        
        
print("Finished at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))

## 2 - Erstellung der info.json

### Basiskonfiguration Config.py

Beispieldaten für ZHB E-codices. Anpassungen können in der config.py vorgenommen werden. 

    collection = 'ZHB E-Codices'
    collection_id = 'zhb_sosa_e-codices'
    ingest_workflow = 'zhb_ecodices'
    keywords = '[E-Codices, ZHB, Sondersammlung]'
    organisation = 'Zentral- und Hochschulbibliothek Luzern'
    organisation_id = 'zhb'
    sets = '[ecodices, zhb, sosa, lara]'
    signature = 'zhb_'

### Signature:

Für die E-Codices-Signaturen werden die DOI verwendet, welche sich aus den Original-Signaturen ableiten lassen. 
Bsp. doi:10.5076/e-codices-zhl-0034-4 / call number:  Msc.34.4 

Signature Beispiel:

    zhb_10_5076_e-codices-zhl-0034-4

   
### Eingabedatei

Die Excel-Datei liegt im working directory. Sie kann relativ leicht in Alma exportiert werden. Die E-codices sind in folgendem Set in der RZS gelistet: 

    e-codices_dlza_heka - Itemized 

Die Export-Datei wurde leicht überarbeitet. Nicht benötigte Spalten werden gelöscht, einige Datenmüssen gesplitted werden:

- Erstellungsdatum aus erster Spalte extrahieren, umbenennen zu Created
- Signatur aus Spalte Availability splitten, umbenennen zu Call number
- MMS_ID als Text erzwingen
- Dateipfade händisch ergänzen. Gibt es mehrere Dateien pro ID, als Array erfassen (bsp. siehe Schillingchronik)
- DOI händisch ergänzen
- ARK händisch ergänzen (derzeit nur bei Schillingchronik vorhanden)

Da die Sammlung überschaubar ist, hält sich der zeitliche Aufwand dafür in Grenzen.


### Export

Für jeden einzelnen record wird eine info.json-Datei erstellt im Format info/signature.json. Das ganze Set wird am Ende noch als json- und Excel-Datei exportiert ins directory 'files' als menschenlesbarer Nachweis, welche Datenobjekte eingelagert wurden. Auch eine Liste aller Signaturen wird als Text-Datei dort abgelegt.


In [ ]:
import json
import config
import pandas as pd
import requests
from datetime import datetime


# Read the Excel file into a pandas DataFrame

input_file = "e-codices.xlsx"
df = pd.read_excel(input_file)

# other variables

completeSet = []
today = datetime.today().strftime('%Y-%m-%d')
sigfile = "files/signatures.txt"

for _, row in df.iterrows():
    
    doi = row['DOI']
    mms_id = str(row['MMS ID'])
    print("MMS ID:", mms_id)
    han_id = row['Record number']
    callno = row['Call number']
    doiurl = config.urldoi+doi
    almaurl = config.urlalma+mms_id
    
    infoSet = {
        'additional': doi.replace('.','_').replace('/','_'),
        'address': config.address, 
        'collection': config.collection,
        'collection_id': config.collection_id ,
        'created': str(row['Created']), 
        'identifiers': [doi, mms_id, han_id, callno],
        'ingest_workflow': config.ingest_workflow, 
        'keywords': config.keywords, 
        'last_changed': today,
        'organisation' : config.organisation,
        'organisation_id' : config.organisation_id,
        'references' : [doiurl, almaurl],
        'signature': config.signature+doi.replace('.','_').replace('/','_'),
        'sets' : config.sets,
        'title' : row['Title'],
        'user' : config.user      
    }
    
    signature = infoSet['signature']
    print("Signature:", signature)
    completeSet.append(infoSet)
    
    # Write the infoSet to a JSON file
    info_json = json.dumps(infoSet, indent=4, ensure_ascii=False)
    
    infofile = f"info/{signature}.json"
    with open(infofile, "w") as outfile:
        outfile.write(info_json)
        print(f"info.json saved as {infofile}")
        
    # write signature to file
    with open(sigfile, 'a') as file:
        file.write(signature)
        file.write("\n")
        print(f"signatures appended to {sigfile}\n")

# Writing completeSet as json file
fulldump = json.dumps(completeSet, indent=4, ensure_ascii=False)
fulljsonfile = "files/ecodices_complete_set.json"
with open(fulljsonfile, "w") as outfile:
    outfile.write(fulldump)
    print(f"---\nAll JSON written to {fulljsonfile}")
    
# Writing completeSet as Excel file
fullexcelfile = "files/ecodices_complete_set.xlsx"
df_json = pd.read_json(fulljsonfile)
df_json.to_excel(fullexcelfile)
print(f"All data saved to Excel file as {fullexcelfile}")


## 3 - Semantische Metadaten

### Metadaten aus Alma (SRU, marcxml)

Mit der alma_id werden die MARC-Daten via SRU aus Alma extrahiert und abgespeichert unter metadata/{signature}/signature.xml. Für gocfl create müssen die Metadaten pro Objekt in einem eigenen Ordner liegen.

### Weitere Metadaten

TODO: Daten aus OAI unifr abholen.

https://www.e-codices.unifr.ch/oai/oai.php?verb=ListRecords&metadataPrefix=oai_dc&set=kol (1 titel)
https://www.e-codices.unifr.ch/oai/oai.php?verb=ListRecords&metadataPrefix=oai_dc&set=zhl (19 titel)


TODO: Zentralgut?


In [ ]:
import requests
import json
import os
from datetime import datetime

url = "https://slsp-rzs.alma.exlibrisgroup.com/view/sru/41SLSP_RZS"
operation = '?version=1.2&operation=searchRetrieve&recordSchema=marcxml&query=rec.id='
sru = url+operation

file_name = 'files/ecodices_complete_set.json'

with open(file_name) as data_file:    
    data = json.load(data_file)
    for value in data:
        # find necessary values
        alma_id = value["identifiers"][1] 
        foldername = value["additional"]
        signature = value["signature"]
        # get SRU response
        query = sru+alma_id
        response = requests.get(query)
        if response.status_code != 200:
            raise Exception(f"SRU request failed with status code {response.status_code}")
        
        # Save the response content as xml to a new directory
        os.mkdir(f'metadata/{foldername}')
        metafile = f"metadata/{foldername}/{signature}.xml"
        
        with open(metafile, 'wb') as file:
            file.write(response.content)
            print(f"\nRecord with ID {alma_id} saved as {metafile}\n---")

print("Finished at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))

       
# TODO get dc Data via e-codices, OAI sets (unifR), or from Zentralgut  


## 4 - Datenobjekte abholen

Stand Juli 2023:
20 Titel, davon 19 ZHB und 1 Korporation Luzern (Schilling Chronik)

Die E-Codices-Objekte liegen auf G:\ZHB-Sosa_Digital\digital unter folgenden Pfaden:

Msc\ecod_Msc...
P\ecod_P...
Romero (Signatur)\ecod_Romero...
S\ecod_...

Die vollständigen Pfade auf G sind in der Eingabedatei ergänzt. Der Dateipfad auf der Workbench wird nach dem DOI benannt und ist in die Infojson unter 'additional' abgelegt. Dieses Script erstellt lediglich die Unterordner mit dem korrekten Namen, die Files werden händisch hierhin verlegt.


In [ ]:
import os

file_name = 'files/ecodices_complete_set.json'

with open(file_name) as data_file:    
    data = json.load(data_file)
    for value in data:
        foldername = value["additional"]
        
        # make a directory for each object
        
        if os.path.exists(f'objects/{foldername}'):
            pass
        else:
            os.mkdir(f'objects/{foldername}')
            
print("Finished creating folder names at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))
